# Terminal Triage UI (`Textual`)

> Interactive terminal triage UI (`TriageApp`) and modal dialogs for inspecting tickets, adding notes, changing statuses (`s`), assigning review apps (`g`), and merging/closing with cleanup (`m` / `x`).

In [ ]:
#| default_exp tui

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import webbrowser
from fastcore.utils import *

from textual.app import App, ComposeResult
from textual.binding import Binding
from textual.containers import Horizontal, Vertical, VerticalScroll
from textual.screen import ModalScreen
from textual.widgets import Button, DataTable, Footer, Header, Input, Label, Select, Static

from craft_support.drive import TicketStore
from craft_support.models import Ticket, TicketStatus, VALID_STATUSES
from craft_support.orchestrator import ReviewOrchestrator

## Modal Dialogs (`NoteModal`, `StatusModal`, `AssignModal`)

- `NoteModal` (`n`): prompts for a triage note and saves it to all selected tickets.
- `StatusModal` (`s`): lets operators quickly transition ticket(s) to any valid lifecycle status (`new`, `triaged`, `closed`, `in_review`, `merged`).
- `AssignModal` (`g`): prompts for `base_branch` (default `main`, or another review branch for stacking) and optional `existing_branch`.

In [ ]:
#| export
class _FormModal(ModalScreen):
    "Base modal with shared layout, focus, and Cancel/Submit button handling."
    DEFAULT_CSS = """
    ModalScreen { align: center middle; }
    .modal-box { width: 64; height: auto; padding: 1 2; border: tall $accent; background: $surface; }
    .modal-actions { height: auto; margin-top: 1; align: right middle; }
    """
    SUBMIT_ID, SUBMIT_LABEL, FOCUS_ID = "save", "Save", ""

    def _actions(self) -> Horizontal:
        return Horizontal(Button('Cancel', id='cancel'), Button(self.SUBMIT_LABEL, variant='primary', id=self.SUBMIT_ID), classes='modal-actions')

    def on_mount(self) -> None:
        if self.FOCUS_ID: self.query_one(self.FOCUS_ID).focus()

    def on_button_pressed(self, event: Button.Pressed) -> None: self.dismiss(self._collect() if event.button.id == self.SUBMIT_ID else None)

class NoteModal(_FormModal):
    "Modal dialog to append a triage note to the selected ticket(s)."
    FOCUS_ID = "#note-input"

    def __init__(self, target_label: str) -> None:
        super().__init__()
        self.target_label = target_label

    def compose(self) -> ComposeResult:
        with Vertical(classes="modal-box"):
            yield Label(f"Add triage note for {self.target_label}")
            yield Input(placeholder="Enter triage note...", id="note-input")
            yield self._actions()

    def _collect(self) -> str | None:
        return self.query_one("#note-input", Input).value.strip() or None

    def on_input_submitted(self, event: Input.Submitted) -> None: self.dismiss(event.value.strip() or None)

class StatusModal(_FormModal):
    "Modal dialog to quickly change the status of selected ticket(s)."
    CSS = ".modal-box { width: 56; }"
    SUBMIT_LABEL, FOCUS_ID = "Update", "#status-select"

    def __init__(self, target_label: str, current_status: TicketStatus = "new") -> None:
        super().__init__()
        store_attr()

    def compose(self) -> ComposeResult:
        default_val = self.current_status if self.current_status in VALID_STATUSES else "triaged"
        with Vertical(classes="modal-box"):
            yield Label(f"Change status for {self.target_label}")
            yield Select([(s, s) for s in VALID_STATUSES], value=default_val, id="status-select", allow_blank=False)
            yield self._actions()

    def _collect(self) -> TicketStatus | None:
        val = str(self.query_one("#status-select", Select).value)
        return val if val in VALID_STATUSES else None  # type: ignore[return-value]

class AssignModal(_FormModal):
    "Modal dialog to assign selected ticket(s) to a new or stacked Solveit review app."
    CSS = ".modal-box { width: 68; }"
    SUBMIT_ID, SUBMIT_LABEL, FOCUS_ID = "confirm", "Assign & Provision", "#base-branch"

    def __init__(self, ticket_ids: list[str]) -> None:
        super().__init__()
        self.ticket_ids = ticket_ids

    def compose(self) -> ComposeResult:
        with Vertical(classes="modal-box"):
            yield Label(f"Assign {len(self.ticket_ids)} ticket(s) to a review app: {', '.join(self.ticket_ids)}")
            yield Label("Base branch (default: main, or parent review-app branch for stacking):")
            yield Input(value="main", placeholder="main", id="base-branch")
            yield Label("Existing branch (optional, attach to an existing review app):")
            yield Input(placeholder="leave empty to create a new branch", id="existing-branch")
            yield self._actions()

    def _collect(self) -> dict[str, str]:
        base = self.query_one("#base-branch", Input).value.strip() or "main"
        existing = self.query_one("#existing-branch", Input).value.strip()
        return dict(base_branch=base, existing_branch=existing)

## Ticket Detail Renderer & `TriageApp`

In [ ]:
#| export
def format_ticket_detail(ticket: Ticket) -> str:
    "Render a readable multi-line summary of a ticket for the TUI detail pane."
    cap = ticket.capture
    vp = f"{cap.viewport.get('width', 0)}x{cap.viewport.get('height', 0)}"
    lines = [f'[bold]{ticket.id}[/bold]  [{ticket.status}]', f'[bold]Title:[/bold] {ticket.title}',
        f'[bold]Updated:[/bold] {ticket.updated_at}  |  [bold]Created:[/bold] {ticket.created_at}', '', '[bold]Description:[/bold]',
        ticket.description or '(none)']
    if ticket.reporter_notes: lines += ["", "[bold]Reporter Notes:[/bold]", ticket.reporter_notes]
    lines += ["", "[bold]Capture Metadata:[/bold]", f"  URL: {cap.url or '(none)'}", f"  Viewport: {vp}",
        f"  Console errors: {len(cap.console_errors)}", *(f"    • {err}" for err in cap.console_errors[:5])]
    lines += [
        "", "[bold]Review & Git Links:[/bold]",
        f"  Group: {ticket.group_id or '-'}",
        f"  Branch: {ticket.branch or '-'} (base: {ticket.base_branch or '-'})",
        f"  PR: #{ticket.pr_number} {ticket.pr_url}" if ticket.pr_number else "  PR: -",
        f"  Review App: {ticket.review_app_id} ({ticket.review_app_url})" if ticket.review_app_id else "  Review App: -",
        f"  Drive Folder: {ticket.drive.folder_url or '-'}",
        f"  DOM Snapshot: {ticket.drive.dom_url or '-'}",
        f"  Screenshot: {ticket.drive.screenshot_url or '-'}",
        "", f"[bold]Triage Notes ({len(ticket.triage_notes)}):[/bold]",
        *([f"  [{n.created_at}] {n.author}: {n.text}" for n in ticket.triage_notes] or ["  (no triage notes yet)"])]
    return "\n".join(lines)

class TriageApp(App[None]):
    "Interactive terminal UI for triaging `solveitcrafts` tickets and managing review apps."
    TITLE = "solveitcrafts Support Triage"
    CSS = """
    #toolbar { height: 3; padding: 0 1; align: left middle; }
    #status-filter { width: 22; }
    #status-banner { margin-left: 2; content-align: left middle; color: $text-muted; }
    #main { height: 1fr; }
    #ticket-table { width: 3fr; height: 1fr; }
    #detail-pane { width: 2fr; height: 1fr; border-left: tall $primary-background; padding: 1 2; }
    """
    BINDINGS = [
        Binding("space", "toggle_select", "Select"),
        Binding("a", "select_all", "Select All"),
        Binding("c", "clear_select", "Clear"),
        Binding("n", "add_note", "Note"),
        Binding("s", "change_status", "Status"),
        Binding("g", "assign_review_app", "Assign App"),
        Binding("o", "open_links", "Open PR/App"),
        Binding("m", "merge_review", "Merge & Cleanup"),
        Binding("x", "close_review", "Close & Cleanup"),
        Binding("r", "refresh_tickets", "Refresh"),
        Binding("q", "quit", "Quit")]

    def __init__(self, store: TicketStore, orchestrator: ReviewOrchestrator | None = None) -> None:
        super().__init__()
        store_attr()
        self.tickets = []
        self.selected_ids = set()
        self.status_filter = None
        self._active_ticket_id = None

    def compose(self) -> ComposeResult:
        yield Header()
        with Horizontal(id="toolbar"):
            yield Select([("All statuses", "all"), *((s, s) for s in VALID_STATUSES)], value="all", id="status-filter", allow_blank=False)
            yield Static("Ready", id="status-banner")
        with Horizontal(id="main"):
            yield DataTable(id="ticket-table", cursor_type="row")
            with VerticalScroll(id="detail-pane"): yield Static("Select a ticket to inspect details.", id="ticket-detail")
        yield Footer()

    async def on_mount(self) -> None:
        self.query_one("#ticket-table", DataTable).add_columns("Sel", "ID", "Status", "Title", "Branch", "PR", "Review App")
        await self.reload_tickets()

### Table Rendering & Selection State

In [ ]:
#| export
@patch
def _set_banner(self: TriageApp, message: str) -> None:
    self.query_one("#status-banner", Static).update(message)

@patch
def _update_selection_banner(self: TriageApp) -> None:
    self._set_banner(f"{len(self.tickets)} ticket(s) | {len(self.selected_ids)} selected")

@patch
async def reload_tickets(self: TriageApp, *, refresh: bool = False) -> None:
    self.tickets = await self.store.list_tickets(status=self.status_filter, refresh=refresh)
    visible_ids = {t.id for t in self.tickets}
    self.selected_ids &= visible_ids
    self._render_table()
    if self.tickets: self._show_ticket(self._active_ticket_id if self._active_ticket_id in visible_ids else self.tickets[0].id)
    else:
        self._active_ticket_id = None
        self.query_one("#ticket-detail", Static).update("No tickets match the current filter.")
    self._update_selection_banner()

@patch
def _render_table(self: TriageApp) -> None:
    table = self.query_one("#ticket-table", DataTable)
    table.clear()
    for t in self.tickets:
        table.add_row(
            "[x]" if t.id in self.selected_ids else "[ ]", t.id, t.status, t.title[:44],
            t.branch or "-", f"#{t.pr_number}" if t.pr_number else "-", t.review_app_id or "-", key=t.id)

@patch
def _ticket_by_id(self: TriageApp, ticket_id: str | None) -> Ticket | None:
    return first(t for t in self.tickets if t.id == ticket_id) if ticket_id else None

@patch
def _current_ticket_id(self: TriageApp) -> str | None:
    table = self.query_one("#ticket-table", DataTable)
    if self.tickets and table.cursor_row is not None and 0 <= table.cursor_row < len(self.tickets): return self.tickets[table.cursor_row].id
    return self._active_ticket_id

@patch
def _target_ticket_ids(self: TriageApp) -> list[str]:
    if self.selected_ids: return [t.id for t in self.tickets if t.id in self.selected_ids]
    cur = self._current_ticket_id()
    return [cur] if cur else []

@patch
def _show_ticket(self: TriageApp, ticket_id: str) -> None:
    self._active_ticket_id = ticket_id
    ticket = self._ticket_by_id(ticket_id)
    self.query_one("#ticket-detail", Static).update(format_ticket_detail(ticket) if ticket else "Ticket not found.")

@patch
async def on_select_changed(self: TriageApp, event: Select.Changed) -> None:
    if event.select.id == "status-filter":
        val = str(event.value)
        self.status_filter = None if val == "all" else val  # type: ignore[assignment]
        await self.reload_tickets()

@patch
def on_data_table_row_highlighted(self: TriageApp, event: DataTable.RowHighlighted) -> None:
    if event.row_key and event.row_key.value: self._show_ticket(str(event.row_key.value))

### Selection & Refresh Actions

In [ ]:
#| export
@patch
def action_toggle_select(self: TriageApp) -> None:
    if not (tid := self._current_ticket_id()): return
    self.selected_ids.symmetric_difference_update({tid})
    self._render_table()
    self._update_selection_banner()

@patch
def action_select_all(self: TriageApp) -> None:
    self.selected_ids = {t.id for t in self.tickets}
    self._render_table()
    self._update_selection_banner()

@patch
def action_clear_select(self: TriageApp) -> None:
    self.selected_ids.clear()
    self._render_table()
    self._update_selection_banner()

@patch
async def action_refresh_tickets(self: TriageApp) -> None:
    await self.reload_tickets(refresh=True)
    self._set_banner(f"Refreshed {len(self.tickets)} ticket(s)")

### Triage, Assignment & Cleanup Actions

In [ ]:
#| export
@patch
def action_add_note(self: TriageApp) -> None:
    targets = self._target_ticket_ids()
    if not targets:
        self._set_banner("No ticket selected.")
        return

    async def _save_note(note: str | None) -> None:
        if not note: return
        try:
            for tid in targets: await self.store.add_triage_note(tid, note)
            await self.reload_tickets()
            self._set_banner(f"Added note to {', '.join(targets)}")
        except Exception as exc: self._set_banner(f"Add note failed: {exc}")

    self.push_screen(NoteModal(", ".join(targets)), _save_note)

@patch
def action_change_status(self: TriageApp) -> None:
    targets = self._target_ticket_ids()
    if not targets:
        self._set_banner("No ticket selected.")
        return
    first_ticket = self._ticket_by_id(targets[0])
    current = first_ticket.status if first_ticket else "new"

    async def _save_status(new_status: TicketStatus | None) -> None:
        if not new_status: return
        try:
            for tid in targets: await self.store.set_status(tid, new_status)
            await self.reload_tickets()
            self._set_banner(f"Set {', '.join(targets)} -> {new_status}")
        except Exception as exc: self._set_banner(f"Status update failed: {exc}")

    self.push_screen(StatusModal(", ".join(targets), current_status=current), _save_status)

@patch
def action_assign_review_app(self: TriageApp) -> None:
    if self.orchestrator is None:
        self._set_banner("Review orchestrator not configured.")
        return
    targets = self._target_ticket_ids()
    if not targets:
        self._set_banner("Select at least one ticket first.")
        return

    async def _do_assign(cfg: dict[str, str] | None) -> None:
        if not cfg: return
        try:
            res = await self.orchestrator.assign_tickets(
                targets, base_branch=cfg.get("base_branch") or "main", existing_branch=cfg.get("existing_branch") or None)
            await self.reload_tickets()
            self._set_banner(f"Assigned {len(targets)} ticket(s) -> PR #{res.pr.number} / {res.review_app.review_app_id}")
        except Exception as exc: self._set_banner(f"Assign failed: {exc}")

    self.push_screen(AssignModal(targets), _do_assign)

@patch
def action_open_links(self: TriageApp) -> None:
    if not (ticket := self._ticket_by_id(self._current_ticket_id())): return
    links = [(lbl, url) for lbl, url in (("PR", ticket.pr_url), ("Review App", ticket.review_app_url), ("Drive", ticket.drive.folder_url)) if url]
    if not links:
        self._set_banner("No PR, review app, or Drive URLs on this ticket.")
        return
    opened, failed = [], []
    for lbl, url in links:
        try: ok = bool(webbrowser.open(url))
        except Exception: ok = False
        (opened if ok else failed).append(lbl)
    parts = ([f"Opened {', '.join(opened)}"] if opened else []) + ([f"Failed to open {', '.join(failed)}"] if failed else [])
    self._set_banner("; ".join(parts))

@patch
async def _complete_selected(self: TriageApp, action: str) -> None:
    if self.orchestrator is None:
        self._set_banner("Review orchestrator not configured.")
        return
    targets = self._target_ticket_ids()
    if not targets:
        self._set_banner("No ticket selected.")
        return
    try:
        res = await self.orchestrator.complete_review(targets, action=action)
        await self.reload_tickets()
        self._set_banner(f"{action.title()}d {', '.join(targets)}; tore down {len(res['torndown_apps'])} review app(s).")
    except Exception as exc: self._set_banner(f"{action.title()} failed: {exc}")

@patch
async def action_merge_review(self: TriageApp) -> None: await self._complete_selected("merge")

@patch
async def action_close_review(self: TriageApp) -> None: await self._complete_selected("close")

In [ ]:
show_doc(TriageApp)

### Headless Textual Pilot Tests

Exercise ticket selection (`a`), triage notes (`n`), status changes (`s`), review-app assignment (`g`), and merge/cleanup (`m`) headlessly via `app.run_test()`.

In [ ]:
import asyncio, subprocess, tempfile
from fastcore.test import test_eq
from craft_support.drive import MemoryTicketStore
from craft_support.orchestrator import FastGitAdapter, GhAdapter, PullRequestInfo, ReviewOrchestrator
from craft_support.solveit_cdp import MemoryReviewAppProvisioner

In [ ]:
#| hide
class _TuiFakeGh(GhAdapter):
    def __init__(self): self.prs, self._n = {}, 10
    async def find_open_pr(self, *, head_branch: str):
        return first(p for p in self.prs.values() if p.branch == head_branch and p.state == "open")
    async def create_pr(self, *, title: str, body: str, head: str, base: str):
        self._n += 1
        pr = PullRequestInfo(number=self._n, url=f"https://github.com/o/r/pull/{self._n}", branch=head, base_branch=base)
        self.prs[pr.number] = pr
        return pr
    async def comment_pr(self, pr_number: int, body: str): pass
    async def merge_pr(self, pr_number: int, *, merge_method: str = "squash"):
        pr = self.prs[pr_number]
        pr.state, pr.merged = "closed", True
        return pr
    async def close_pr(self, pr_number: int):
        pr = self.prs[pr_number]
        pr.state, pr.merged = "closed", False
        return pr

async def _test_tui_pilot():
    with tempfile.TemporaryDirectory() as td:
        repo = Path(td)/"repo"
        repo.mkdir()
        subprocess.run(["git", "init", "-b", "main"], cwd=repo, check=True, capture_output=True)
        subprocess.run(["git", "config", "user.email", "t@e.com"], cwd=repo, check=True)
        subprocess.run(["git", "config", "user.name", "T"], cwd=repo, check=True)
        (repo/"README.md").write_text("# t\n", encoding="utf-8")
        subprocess.run(["git", "add", "README.md"], cwd=repo, check=True)
        subprocess.run(["git", "commit", "-m", "init"], cwd=repo, check=True, capture_output=True)

        store, git, prov = MemoryTicketStore(), FastGitAdapter(repo, push_enabled=False), MemoryReviewAppProvisioner()
        orch = ReviewOrchestrator(store=store, git=git, gh=_TuiFakeGh(), provisioner=prov)

        t1, _ = await store.create_ticket(title="TUI ticket 1", description="First bug")
        t2, _ = await store.create_ticket(title="TUI ticket 2", description="Second bug")
        assert "TUI ticket 1" in format_ticket_detail(t1)

        app = TriageApp(store=store, orchestrator=orch)
        async with app.run_test() as pilot:
            await pilot.pause()
            test_eq(len(app.tickets), 2)

            await pilot.press("a")
            test_eq(app.selected_ids, {t1.id, t2.id})
            await pilot.press("n")
            await pilot.pause()
            await pilot.press(*list("Grouped"), "enter")
            await pilot.pause()
            test_eq((await store.get_ticket(t1.id)).status, "triaged")

            await pilot.press("c")
            await pilot.press("s")
            await pilot.pause()
            app.screen.query_one("#status-select", Select).value = "closed"
            await pilot.click("#save")
            await pilot.pause()
            test_eq((await store.get_ticket(app.tickets[0].id)).status, "closed")

            await pilot.press("a")
            await pilot.press("g")
            await pilot.pause()
            await pilot.click("#confirm")
            await pilot.pause()
            t1_after = await store.get_ticket(t1.id)
            assert t1_after.review_app_id in prov.instances

            await pilot.press("a")
            await pilot.press("m")
            await pilot.pause()
            test_eq((await store.get_ticket(t1.id)).status, "merged")
            assert t1_after.review_app_id not in prov.instances

await _test_tui_pilot()